# Phase 1: Our Own Camera Poses with COLMAP (garden)

In Phase 0 we trained on the **official** Mip-NeRF 360 poses. Real projects (your own videos, DL3DV, Wayve) don't come with poses, so here we **compute them ourselves** and check how good they are.

**Pipeline** (the same one as the official 3DGS `convert.py`):
`feature_extractor (SIFT, GPU)` → `exhaustive_matcher` → `mapper` (incremental SfM + bundle adjustment) → `image_undistorter` → gsplat training
Optional: **global SfM** (`colmap global_mapper`, GLOMAP's method) on the same matches, if this COLMAP has it, for a speed/accuracy comparison.

**What we measure**
| Check | How | Target |
|---|---|---|
| Registration | images registered / total | > 95 % |
| Reprojection error | `colmap model_analyzer` | < ~1 px |
| Pose accuracy | align our cameras to the official ones (Umeyama similarity), then rotation error (°) + camera-center error (% of scene radius) | median < 0.5° and < 2 % |
| Does it matter? | train **official** vs **our** poses on the **same undistorted images** at 1/4 resolution, 7k steps | PSNR within ~0.3 dB |

**Before running:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.
The install cell may **restart the runtime once** (numpy change). If that happens, just `Run all` again: finished steps are skipped.

⏱️ Rough total on a T4: ~60–80 min (most of it is the two training runs). SfM results are cached on Drive, so re-runs are fast.

## 1. GPU, Drive, paths

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
from google.colab import drive
drive.mount('/content/drive')

import os, sys, glob, json, time, re, subprocess, shutil
PROJECT = '/content/drive/MyDrive/3dgs-digital-twin'
WORK    = '/content/work'
OUT     = f'{PROJECT}/results/phase1_garden_sfm'          # pose QA outputs + cached sparse models
for d in [f'{PROJECT}/results', f'{PROJECT}/env', f'{PROJECT}/cache/torch_extensions', WORK, OUT]:
    os.makedirs(d, exist_ok=True)

# ---- knobs ----
SCENE = 'garden'
STEPS = 7000                    # quick comparison runs (Phase 2 does 30k)
RUN_GLOMAP = True               # also run global SfM for comparison
TRAIN_OFFICIAL_BASELINE = True  # train on official poses at 1/4 res for a fair comparison

## 2. Install gsplat + trainer deps (with all Phase 0 fixes)

Same as Phase 0, with the fixes from `docs/failures.md` F0.1–F0.5 built in. The cell skips anything already done, so it's safe to re-run.

In [ ]:
import importlib.util
MARKER = '/content/.gs_deps_ok'          # /content survives a kernel restart (not a VM reset)

if not os.path.exists(MARKER):
    import torch
    np_before = subprocess.run([sys.executable, '-c', 'import numpy; print(numpy.__version__)'],
                               capture_output=True, text=True).stdout.strip()
    # gsplat: prebuilt wheel if available, else PyPI (JIT compile)
    tv = torch.__version__.split('+')[0].split('.'); cu = (torch.version.cuda or '').replace('.', '')
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gsplat==1.5.3',
                        '--index-url', f'https://docs.gsplat.studio/whl/pt{tv[0]}{tv[1]}cu{cu}',
                        '--extra-index-url', 'https://pypi.org/simple'], capture_output=True, text=True)
    if r.returncode != 0:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gsplat==1.5.3'], check=True)

    REPO = f'{WORK}/gsplat'
    if not os.path.exists(REPO):
        subprocess.run(['git', 'clone', '-q', '--depth', '1', '--branch', 'v1.5.3',
                        'https://github.com/nerfstudio-project/gsplat.git', REPO], check=True)
    reqs = open(f'{REPO}/examples/requirements.txt').read()

    # F0.1 + F0.2: git deps (several glued onto one line) -> split, build without isolation
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ninja', 'setuptools', 'wheel'], check=True)
    GPU_NAME = torch.cuda.get_device_name(0)
    if 'T4' in GPU_NAME:
        os.environ['TORCH_CUDA_ARCH_LIST'] = '7.5'   # faster builds, T4 only
    for url in re.findall(r'git\+https://github\.com/[^@\s]+@[0-9a-f]{40}', reqs):
        print('building', url.split('/')[-1][:60])
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation', url], check=True)

    # F0.3: drop git lines + numpy<2 pin, install the rest
    rest = [l for l in reqs.splitlines() if 'git+https' not in l and not l.lower().startswith('numpy')]
    open('/tmp/reqs.txt', 'w').write('\n'.join(rest))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', '/tmp/reqs.txt'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'remotezip', 'plyfile', 'numpy>=2.1,<2.5'], check=True)

    # F0.5: pycolmap fork uses np.uint64(-1) -> OverflowError on numpy 2 (patch the file, don't import it)
    f = os.path.join(os.path.dirname(importlib.util.find_spec('pycolmap').origin), 'scene_manager.py')
    s = open(f).read()
    if 'np.uint64(-1)' in s:
        open(f, 'w').write(s.replace('np.uint64(-1)', 'np.uint64(18446744073709551615)')); print('patched pycolmap')

    # F1.5: otherwise gsplat recompiles its CUDA code at the first render of every session (~13 min).
    # Build a compiled package once, keep it on Drive, and install that in later sessions.
    if not subprocess.run([sys.executable, '-c', 'from gsplat import csrc'], capture_output=True).returncode == 0:
        WHEELS = (f"{PROJECT}/cache/wheels/gsplat153_pt{torch.__version__.split('+')[0]}_cu{cu}"
                  f"_py{sys.version_info.major}{sys.version_info.minor}_{GPU_NAME.replace(' ', '-')}")
        os.makedirs(WHEELS, exist_ok=True)
        if not glob.glob(f'{WHEELS}/gsplat-*.whl'):
            print('Compiling gsplat once for this GPU + torch (10-20 min). Later sessions reuse it.')
            r = subprocess.run([sys.executable, '-m', 'pip', 'wheel', '-q', '--no-deps', '--no-build-isolation', '-w', WHEELS,
                                'git+https://github.com/nerfstudio-project/gsplat.git@v1.5.3'],
                               capture_output=True, text=True)
            if r.returncode != 0:
                print('Building failed, gsplat will compile during training instead:', r.stderr[-800:])
        for whl in glob.glob(f'{WHEELS}/gsplat-*.whl'):
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', whl], check=True)
    print('gsplat precompiled:', subprocess.run([sys.executable, '-c', 'from gsplat import csrc'], capture_output=True).returncode == 0)

    # F0.4: make gsplat's examples/datasets a regular package (beats HF `datasets`)
    open(f'{REPO}/examples/datasets/__init__.py', 'a').close()

    open(MARKER, 'w').write('ok')
    np_after = subprocess.run([sys.executable, '-c', 'import numpy; print(numpy.__version__)'],
                              capture_output=True, text=True).stdout.strip()
    if np_before != np_after:
        print(f'numpy {np_before} -> {np_after}: restarting runtime. Then "Run all" again.')
        time.sleep(2); os.kill(os.getpid(), 9)

REPO = f'{WORK}/gsplat'
check = subprocess.run([sys.executable, '-c',
    'import numpy, gsplat, fused_ssim, pycolmap, tyro; print("numpy", numpy.__version__, "| gsplat", gsplat.__version__)'],
    capture_output=True, text=True, cwd=f'{REPO}/examples')
print(check.stdout or check.stderr[-2000:])

## 3. Install COLMAP in a separate environment

We use **micromamba + conda-forge** and pin the **CUDA build** of COLMAP 4.2.1 plus GPU `libfaiss` (GPU SIFT extraction and matching).
It lives in its own environment (`/content/work/sfm`), so it can't clash with the `pycolmap` fork gsplat uses.

In [ ]:
MAMBA   = f'{WORK}/bin/micromamba'
SFM_ENV = f'{WORK}/sfm'
os.environ['MAMBA_ROOT_PREFIX'] = f'{WORK}/mamba'
if not os.path.exists(MAMBA):
    subprocess.run(f'curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C {WORK} bin/micromamba',
                   shell=True, check=True)

COLMAP = f'{SFM_ENV}/bin/colmap'
SFM_ENVVARS = {**os.environ, 'QT_QPA_PLATFORM': 'offscreen'}

def colmap_banner():
    r = subprocess.run([COLMAP, '-h'], capture_output=True, text=True, env=SFM_ENVVARS)
    return r.returncode, r.stdout + r.stderr

# Known-good combo (see docs/failures.md F1.1). Don't add the separate `glomap` package: it breaks this build.
if not os.path.exists(COLMAP) or colmap_banner()[0] != 0:
    shutil.rmtree(SFM_ENV, ignore_errors=True); t = time.time()
    subprocess.run([MAMBA, 'create', '-y', '-q', '-p', SFM_ENV, '-c', 'conda-forge',
                    'colmap=4.2.1=*cuda*', 'libfaiss=*=*cuda*'], check=True)
    print(f'installed in {time.time()-t:.0f}s')

rc, hdr = colmap_banner()
print('\n'.join(hdr.splitlines()[:2]))
if rc != 0:
    raise RuntimeError('COLMAP does not start, see the error above')   # never mistake this for "CPU-only"
COLMAP_CUDA = 'with CUDA' in hdr
# COLMAP 4 may include GLOMAP's global SfM as a built-in command
HAS_GLOBAL_MAPPER = subprocess.run([COLMAP, 'global_mapper', '-h'], capture_output=True, env=SFM_ENVVARS).returncode == 0
print('COLMAP CUDA:', COLMAP_CUDA, '| global_mapper:', HAS_GLOBAL_MAPPER)
if not COLMAP_CUDA:
    print('⚠️ COLMAP runs but without CUDA: SIFT + exhaustive matching will be slow (~30-60 min).')

In [ ]:
# helpers: run a command with logging, and pick option names that exist in this COLMAP version
SFM_LOG = f'{OUT}/sfm_log.txt'

def sh(cmd, tail=12, cwd=None):
    t = time.time()
    p = subprocess.run([str(c) for c in cmd], capture_output=True, text=True, env=SFM_ENVVARS, cwd=cwd)
    out = p.stdout + p.stderr
    with open(SFM_LOG, 'a') as f:
        f.write(f'\n$ {" ".join(map(str, cmd))}\n{out}\n')
    dt = time.time() - t
    lines = [l for l in out.strip().splitlines() if l.strip()]
    if p.returncode != 0:
        print('\n'.join(lines[-40:]))
        raise RuntimeError(f'{os.path.basename(str(cmd[0]))} {cmd[1]} failed (exit {p.returncode}), full log: {SFM_LOG}')
    if tail: print('\n'.join(lines[-tail:]))
    print(f'✔ {os.path.basename(str(cmd[0]))} {cmd[1]}: {dt/60:.1f} min')
    return out, dt

_help_cache = {}
def opt(sub, *candidates):
    # COLMAP renamed some options between versions (e.g. SiftExtraction.use_gpu -> FeatureExtraction.use_gpu)
    if sub not in _help_cache:
        r = subprocess.run([COLMAP, sub, '-h'], capture_output=True, text=True, env=SFM_ENVVARS)
        _help_cache[sub] = r.stdout + r.stderr
    for c in candidates:
        if f'--{c}' in _help_cache[sub]:
            return c
    return None
print('extract gpu opt:', opt('feature_extractor', 'FeatureExtraction.use_gpu', 'SiftExtraction.use_gpu'))
print('match gpu opt:  ', opt('exhaustive_matcher', 'FeatureMatching.use_gpu', 'SiftMatching.use_gpu'))

## 4. Download garden (official poses + 1/4-res images)

We pull only `sparse/` (the official poses, our "ground truth") and `images_4/` (~1297×840 px). We run SfM on the 1/4-res images: faster, and plenty for accurate poses.

In [ ]:
from remotezip import RemoteZip
URL = 'http://storage.googleapis.com/gresearch/refraw360/360_v2.zip'
OFFICIAL = f'{WORK}/data/360_v2/{SCENE}'
WANT = ('sparse/', 'poses_bounds.npy', 'images_4/')
if not os.path.exists(f'{OFFICIAL}/images_4'):
    t = time.time()
    with RemoteZip(URL) as z:
        members = [m for m in z.namelist() if f'{SCENE}/' in m and not m.endswith('/')
                   and any(m.split(f'{SCENE}/', 1)[1].startswith(w) for w in WANT)]
        for m in members:
            dst = f'{OFFICIAL}/' + m.split(f'{SCENE}/', 1)[1]
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            with z.open(m) as src, open(dst, 'wb') as f:
                f.write(src.read())
    print(f'downloaded {len(members)} files in {time.time()-t:.0f}s')
if not os.path.exists(f'{OFFICIAL}/images'):
    os.symlink(f'{OFFICIAL}/images_4', f'{OFFICIAL}/images')
# ⚠️ Train this folder with --data_factor 1. With factor > 1 and JPGs, gsplat downscales images/ by that factor
# again (docs/failures.md F1.2). With factor 1 it scales the full-res intrinsics to the real image size itself.
IMAGES = f'{OFFICIAL}/images_4'
print(len(os.listdir(IMAGES)), 'images in', IMAGES)

## 5. Image-quality check (blur score)

The variance of the Laplacian is a cheap sharpness score: low means blurry. Mip-NeRF 360 is curated, so we **only report** here. This is the tool we'll use to **drop** bad frames from raw video later (DL3DV / Wayve / your own captures).

In [ ]:
import numpy as np, imageio.v2 as imageio, matplotlib.pyplot as plt
def blur_score(path):
    g = imageio.imread(path).astype(np.float32)
    g = g[..., :3].mean(-1) if g.ndim == 3 else g
    lap = g[1:-1, 2:] + g[1:-1, :-2] + g[2:, 1:-1] + g[:-2, 1:-1] - 4 * g[1:-1, 1:-1]
    return float(lap.var())
names = sorted(os.listdir(IMAGES))
scores = np.array([blur_score(f'{IMAGES}/{n}') for n in names])
thr = np.percentile(scores, 10)
print(f'blur score: min {scores.min():.0f} | median {np.median(scores):.0f} | max {scores.max():.0f}')
print('10% blurriest (would be dropped on raw video):', [names[i] for i in np.argsort(scores)[:5]], '...')
plt.figure(figsize=(12, 2.5)); plt.plot(scores, '.-'); plt.axhline(thr, color='r', ls='--', label='10th pct')
plt.xlabel('frame'); plt.ylabel('Laplacian var'); plt.legend(); plt.title('Sharpness per frame'); plt.show()

## 6. Run COLMAP

`exhaustive_matcher` matches every image pair (185 images ≈ 17k pairs). That's right for unordered photos like these. For long videos we'll switch to `sequential_matcher`.
The sparse model is cached on Drive (`results/phase1_garden_sfm/sparse_colmap`). Delete that folder to force a re-run.

In [ ]:
SFM = f'{WORK}/sfm_{SCENE}'
DB  = f'{SFM}/database.db'
os.makedirs(SFM, exist_ok=True)
CACHE_COLMAP = f'{OUT}/sparse_colmap'
timings = json.load(open(f'{OUT}/timings.json')) if os.path.exists(f'{OUT}/timings.json') else {}

def n_images_in(model_dir):
    import struct
    with open(f'{model_dir}/images.bin', 'rb') as f:
        return struct.unpack('<Q', f.read(8))[0]

if os.path.exists(f'{CACHE_COLMAP}/images.bin'):
    print('✔ using cached COLMAP model from Drive'); BEST = CACHE_COLMAP
else:
    open(SFM_LOG, 'w').close()
    if os.path.exists(DB): os.remove(DB)
    gpu_e = opt('feature_extractor', 'FeatureExtraction.use_gpu', 'SiftExtraction.use_gpu')
    gpu_m = opt('exhaustive_matcher', 'FeatureMatching.use_gpu', 'SiftMatching.use_gpu')
    G = '1' if COLMAP_CUDA else '0'

    cmd = [COLMAP, 'feature_extractor', '--database_path', DB, '--image_path', IMAGES,
           '--ImageReader.single_camera', '1', '--ImageReader.camera_model', 'OPENCV']
    if gpu_e: cmd += [f'--{gpu_e}', G]
    _, timings['extract_s'] = sh(cmd)

    cmd = [COLMAP, 'exhaustive_matcher', '--database_path', DB]
    if gpu_m: cmd += [f'--{gpu_m}', G]
    _, timings['match_s'] = sh(cmd)

    os.makedirs(f'{SFM}/sparse_raw', exist_ok=True)
    cmd = [COLMAP, 'mapper', '--database_path', DB, '--image_path', IMAGES, '--output_path', f'{SFM}/sparse_raw']
    tol = opt('mapper', 'Mapper.ba_global_function_tolerance')
    if tol: cmd += [f'--{tol}', '0.000001']          # same as official 3DGS convert.py
    _, timings['colmap_mapper_s'] = sh(cmd, tail=5)

    models = sorted(glob.glob(f'{SFM}/sparse_raw/*/images.bin'))
    print('models found:', {os.path.dirname(m).split('/')[-1]: n_images_in(os.path.dirname(m)) for m in models})
    BEST = max((os.path.dirname(m) for m in models), key=n_images_in)
    shutil.copytree(BEST, CACHE_COLMAP, dirs_exist_ok=True)
    json.dump(timings, open(f'{OUT}/timings.json', 'w'), indent=1)
print('COLMAP model:', BEST, '|', n_images_in(BEST), '/', len(os.listdir(IMAGES)), 'images registered')

### 6b. (Optional) Global SfM on the same matches

Global SfM (the GLOMAP approach) solves all camera rotations and positions at once instead of adding images one at a time. It's usually much faster than `colmap mapper` at similar accuracy, and worth knowing about for big scenes.

In [ ]:
CACHE_GLOMAP = f'{OUT}/sparse_glomap'
GLOMAP_MODEL = None
if RUN_GLOMAP and HAS_GLOBAL_MAPPER:
    if os.path.exists(f'{CACHE_GLOMAP}/images.bin'):
        print('✔ using cached global SfM model'); GLOMAP_MODEL = CACHE_GLOMAP
    elif os.path.exists(DB):
        try:
            os.makedirs(f'{SFM}/sparse_glomap', exist_ok=True)
            _, timings['glomap_mapper_s'] = sh([COLMAP, 'global_mapper', '--database_path', DB, '--image_path', IMAGES,
                                                '--output_path', f'{SFM}/sparse_glomap'], tail=5)
            model = os.path.dirname(max(glob.glob(f'{SFM}/sparse_glomap/**/images.bin', recursive=True),
                                        key=lambda m: n_images_in(os.path.dirname(m))))
            shutil.copytree(model, CACHE_GLOMAP, dirs_exist_ok=True)
            GLOMAP_MODEL = CACHE_GLOMAP
            json.dump(timings, open(f'{OUT}/timings.json', 'w'), indent=1)
        except Exception as e:
            print('Global SfM failed (optional, continuing):', e)
    else:
        print('No database (COLMAP model came from cache). Delete the Drive cache to re-run both mappers.')
else:
    print('Global SfM skipped (RUN_GLOMAP=False or this COLMAP has no global_mapper)')
print('timings (min):', {k: round(v/60, 1) for k, v in timings.items()})

## 7. Pose QA, part 1: COLMAP's own statistics

In [ ]:
def analyze(model_dir, label):
    out, _ = sh([COLMAP, 'model_analyzer', '--path', model_dir], tail=0)
    def grab(key):
        m = re.search(key + r'[^:]*:\s*([0-9.]+)', out)
        return float(m.group(1)) if m else None
    st = {'model': label,
          'registered': n_images_in(model_dir),   # read from the model, not from log text
          'total_images': len(os.listdir(IMAGES)),
          'points3D': int.from_bytes(open(f'{model_dir}/points3D.bin', 'rb').read(8), 'little'),
          'mean_track_length': grab('Mean track length'),
          'mean_obs_per_image': grab('Mean observations per image'),
          'mean_reproj_error_px': grab('Mean reprojection error')}
    st['registered_pct'] = round(100 * st['registered'] / st['total_images'], 1)
    return st

qa_stats = [analyze(BEST, 'colmap')]
if GLOMAP_MODEL: qa_stats.append(analyze(GLOMAP_MODEL, 'glomap'))
for st in qa_stats:
    print(json.dumps(st, indent=1))
    ok_reg = st['registered_pct'] > 95
    ok_rep = st['mean_reproj_error_px'] is not None and st['mean_reproj_error_px'] < 1.0
    print(f"  registered > 95%: {'✅' if ok_reg else '❌'}   reprojection < 1px: {'✅' if ok_rep else '❌'}")

## 8. Pose QA, part 2: compare with the official poses

An SfM reconstruction is only defined **up to a similarity transform** (scale, rotation, translation). So we:
1. match cameras by file name,
2. find the best similarity transform from our camera centers to the official ones (**Umeyama** alignment),
3. measure each camera's **rotation error** (degrees) and **center error** (as % of the scene radius = median camera distance from the scene center).

In [ ]:
import struct
def read_images_bin(path):
    imgs = {}
    with open(path, 'rb') as f:
        n = struct.unpack('<Q', f.read(8))[0]
        for _ in range(n):
            struct.unpack('<i', f.read(4))                       # image_id
            q = np.array(struct.unpack('<4d', f.read(32)))        # qw qx qy qz (world -> camera)
            t = np.array(struct.unpack('<3d', f.read(24)))
            struct.unpack('<i', f.read(4))                       # camera_id
            name = b''
            while (c := f.read(1)) != b'\x00':
                name += c
            n2d = struct.unpack('<Q', f.read(8))[0]
            f.seek(24 * n2d, 1)                                   # skip 2D points (x, y, point3D_id)
            imgs[os.path.basename(name.decode())] = (q, t)
    return imgs

def qvec2rotmat(q):
    w, x, y, z = q / np.linalg.norm(q)
    return np.array([[1-2*y*y-2*z*z, 2*x*y-2*w*z,   2*x*z+2*w*y],
                     [2*x*y+2*w*z,   1-2*x*x-2*z*z, 2*y*z-2*w*x],
                     [2*x*z-2*w*y,   2*y*z+2*w*x,   1-2*x*x-2*y*y]])

def poses(imgs, names):
    R = np.stack([qvec2rotmat(imgs[n][0]) for n in names])
    C = np.stack([-R[i].T @ imgs[n][1] for i, n in enumerate(names)])  # camera centers in world
    return R, C

def umeyama(src, dst):
    # find s, R, t minimising || dst - (s R src + t) ||
    mu_s, mu_d = src.mean(0), dst.mean(0)
    xs, xd = src - mu_s, dst - mu_d
    U, D, Vt = np.linalg.svd(xd.T @ xs / len(src))
    S = np.eye(3)
    if np.linalg.det(U) * np.linalg.det(Vt) < 0: S[2, 2] = -1
    R = U @ S @ Vt
    s = np.trace(np.diag(D) @ S) / (xs ** 2).sum(1).mean()
    return s, R, mu_d - s * R @ mu_s

def compare(model_dir, label, official):
    ours = read_images_bin(f'{model_dir}/images.bin')
    names = sorted(set(ours) & set(official))
    R_o, C_o = poses(ours, names)
    R_g, C_g = poses(official, names)
    s, Ra, ta = umeyama(C_o, C_g)
    C_al = (s * (Ra @ C_o.T)).T + ta
    R_al = R_o @ Ra.T                                            # our camera rotations in the official frame
    cosang = (np.einsum('nij,nij->n', R_g, R_al) - 1) / 2         # trace(R_g^T R_al) for each camera
    rot_err = np.degrees(np.arccos(np.clip(cosang, -1, 1)))
    radius = np.median(np.linalg.norm(C_g - C_g.mean(0), axis=1))
    ctr_err = 100 * np.linalg.norm(C_al - C_g, axis=1) / radius
    res = {'model': label, 'matched_cameras': len(names),
           'rot_err_deg_median': float(np.median(rot_err)), 'rot_err_deg_mean': float(rot_err.mean()),
           'rot_err_deg_max': float(rot_err.max()),
           'center_err_pct_median': float(np.median(ctr_err)), 'center_err_pct_mean': float(ctr_err.mean()),
           'center_err_pct_max': float(ctr_err.max())}
    return res, dict(names=names, C_g=C_g, C_al=C_al, rot_err=rot_err, ctr_err=ctr_err)

official = read_images_bin(f'{OFFICIAL}/sparse/0/images.bin')
pose_results, pose_details = [], {}
for model_dir, label in [(BEST, 'colmap')] + ([(GLOMAP_MODEL, 'glomap')] if GLOMAP_MODEL else []):
    r, d = compare(model_dir, label, official)
    pose_results.append(r); pose_details[label] = d
    ok = r['rot_err_deg_median'] < 0.5 and r['center_err_pct_median'] < 2
    print(f"{label:>7}: rotation err median {r['rot_err_deg_median']:.3f}° (max {r['rot_err_deg_max']:.2f}°) | "
          f"center err median {r['center_err_pct_median']:.2f}% (max {r['center_err_pct_max']:.2f}%)  {'✅' if ok else '⚠️'}")

In [ ]:
# Plots: camera layout (official vs ours, aligned) + error per camera
d = pose_details['colmap']
Cg = d['C_g'] - d['C_g'].mean(0)
_, _, V = np.linalg.svd(Cg, full_matrices=False)          # project onto the plane of the camera ring
P = lambda X: (X - d['C_g'].mean(0)) @ V[:2].T
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
ax[0].scatter(*P(d['C_g']).T, s=30, facecolors='none', edgecolors='k', label='official')
for lbl, mk, col in [('colmap', 'x', 'tab:red'), ('glomap', '+', 'tab:blue')]:
    if lbl in pose_details:
        ax[0].scatter(*P(pose_details[lbl]['C_al']).T, s=18, marker=mk, c=col, label=f'{lbl} (aligned)')
ax[0].set_aspect('equal'); ax[0].legend(); ax[0].set_title('Camera centers (top-down)')
for lbl, col in [('colmap', 'tab:red'), ('glomap', 'tab:blue')]:
    if lbl in pose_details:
        ax[1].plot(pose_details[lbl]['rot_err'], '.-', c=col, label=lbl)
        ax[2].plot(pose_details[lbl]['ctr_err'], '.-', c=col, label=lbl)
ax[1].set_title('Rotation error per camera (°)'); ax[1].set_xlabel('camera (sorted by name)'); ax[1].legend()
ax[2].set_title('Center error per camera (% of scene radius)'); ax[2].set_xlabel('camera'); ax[2].legend()
plt.tight_layout(); plt.savefig(f'{OUT}/pose_comparison.png', dpi=110); plt.show()

json.dump({'sfm_stats': qa_stats, 'pose_vs_official': pose_results, 'timings_s': timings},
          open(f'{OUT}/pose_qa.json', 'w'), indent=1)
print('saved', f'{OUT}/pose_qa.json', 'and pose_comparison.png')

## 9. Undistort → training-ready scene

We solved for an **OPENCV** camera (with lens distortion). `image_undistorter` produces distortion-free **PINHOLE** images plus a matching sparse model, the exact format the 3DGS trainers expect.

In [ ]:
OURS = f'{WORK}/data/{SCENE}_ours'
if not os.path.exists(f'{OURS}/sparse/0/images.bin'):
    shutil.rmtree(OURS, ignore_errors=True)
    sh([COLMAP, 'image_undistorter', '--image_path', IMAGES, '--input_path', BEST,
        '--output_path', OURS, '--output_type', 'COLMAP'], tail=3)
    os.makedirs(f'{OURS}/sparse/0', exist_ok=True)
    for fn in glob.glob(f'{OURS}/sparse/*.bin'):
        shutil.move(fn, f'{OURS}/sparse/0/')
print(len(os.listdir(f'{OURS}/images')), 'undistorted images | size', imageio.imread(glob.glob(f'{OURS}/images/*')[0]).shape)

## 10. Train: official poses vs our poses (same resolution, same steps)

- **official**: Mip-NeRF 360 poses, `images_4` as `images/` (`--data_factor 1`)
- **ours**: our COLMAP poses, undistorted 1/4-res images (`--data_factor 1`)

Both folders already hold 1/4-res images, so both use factor 1. `train()` checks the size of the rendered images and stops if it doesn't match.

⚠️ **These two runs are scored against different images** (original JPGs vs undistorted), so their PSNRs aren't comparable (`docs/failures.md` F1.3). The fair test is the third run:

- **official_ourimages**: official poses + our undistorted images + our undistorted camera. Compared with **ours**, the poses are the only difference.

Both use the same held-out test views (every 8th image) and **VGG LPIPS** (paper-comparable, unlike Phase 0's AlexNet default).
Each run prints a progress line about once a minute.

In [ ]:
help_txt = subprocess.run([sys.executable, 'simple_trainer.py', 'default', '--help'], capture_output=True,
                          text=True, cwd=f'{REPO}/examples').stdout
LPIPS = ['--lpips_net', 'vgg'] if re.search(r'lpips.net', help_txt) else []
print('LPIPS flag:', LPIPS or 'not available (AlexNet default)')

def check_resolution(rd, data_dir):
    # eval renders are GT | render side by side, so each half must match the input image width
    expected = imageio.imread(sorted(f for f in glob.glob(f'{data_dir}/images/**/*', recursive=True) if os.path.isfile(f))[0]).shape[1]
    actual = imageio.imread(sorted(glob.glob(f'{rd}/renders/val_step*_*.png'))[0]).shape[1] / 2
    print(f'   trained at width {actual:.0f}px (input images: {expected}px)')
    if abs(actual - expected) / expected > 0.1:
        raise RuntimeError(f'Trained at {actual:.0f}px instead of {expected}px: wrong data_factor? See F1.2')

def train(run, data_dir, factor):
    rd = f'{PROJECT}/results/{run}'
    stats = sorted(glob.glob(f'{rd}/stats/val_step*.json'))
    if stats:
        print(f'✔ {run}: already trained, loading stats'); check_resolution(rd, data_dir)
        s = json.load(open(stats[-1])); s['train_min'] = None; return s
    cmd = [sys.executable, 'simple_trainer.py', 'default', '--data_dir', data_dir, '--data_factor', str(factor),
           '--result_dir', rd, '--max_steps', str(STEPS), '--eval_steps', str(STEPS), '--save_steps', str(STEPS),
           '--disable_viewer', *LPIPS, *globals().get('TRAIN_EXTRA', [])]
    os.makedirs(rd, exist_ok=True)
    t0 = last = time.time(); buf = ''; tail = []
    p = subprocess.Popen(cmd, cwd=f'{REPO}/examples', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    with open(f'{rd}/train_log.txt', 'w') as lf:
        while (chunk := p.stdout.read(512)):
            lf.write(chunk); buf += chunk
            parts = re.split(r'[\r\n]', buf); buf = parts[-1]
            for ln in filter(str.strip, parts[:-1]):
                tail = (tail + [ln])[-40:]
                if time.time() - last > 60:
                    print(f'[{(time.time()-t0)/60:4.1f} min] {ln.strip()[:140]}'); last = time.time()
    if p.wait() != 0:
        print('\n'.join(tail)); raise RuntimeError(f'training failed, log: {rd}/train_log.txt')
    s = json.load(open(sorted(glob.glob(f'{rd}/stats/val_step*.json'))[-1]))
    s['train_min'] = round((time.time() - t0) / 60, 1)
    check_resolution(rd, data_dir)
    print(f"✔ {run}: PSNR {s['psnr']:.2f} | SSIM {s['ssim']:.3f} | LPIPS {s['lpips']:.3f} | "
          f"{s.get('num_GS', 0):,} GS | {s['train_min']} min")
    return s

runs = {}
if TRAIN_OFFICIAL_BASELINE:
    runs['official'] = train(f'phase1_{SCENE}_1of4_{STEPS//1000}k_official_fixed', OFFICIAL, 1)
    # Controlled test: official poses on our images, using our undistorted camera
    SWAP = f'{WORK}/data/{SCENE}_officialposes_ourimages'
    if not os.path.exists(f'{SWAP}/sparse/0/images.bin'):
        os.makedirs(f'{SWAP}/sparse/0', exist_ok=True)
        if not os.path.exists(f'{SWAP}/images'):
            os.symlink(f'{OURS}/images', f'{SWAP}/images')
        for fn in ['images.bin', 'points3D.bin']:
            shutil.copy(f'{OFFICIAL}/sparse/0/{fn}', f'{SWAP}/sparse/0/{fn}')
        off = open(f'{OFFICIAL}/sparse/0/cameras.bin', 'rb').read()
        cam = bytearray(open(f'{OURS}/sparse/0/cameras.bin', 'rb').read())
        assert struct.unpack_from('<Q', off)[0] == 1 and struct.unpack_from('<Q', cam)[0] == 1, 'expected one camera'
        cam[8:12] = off[8:12]                       # use the camera id the official poses refer to
        open(f'{SWAP}/sparse/0/cameras.bin', 'wb').write(cam)
    runs['official_ourimages'] = train(f'phase1_{SCENE}_1of4_{STEPS//1000}k_officialposes_ourimages', SWAP, 1)
runs['ours'] = train(f'phase1_{SCENE}_f4_{STEPS//1000}k_ourposes', OURS, 1)   # factor 1: this run was always true 1/4 res

## 11. Results

In [ ]:
import csv, datetime, torch
rows = []
for k, s in runs.items():
    rows.append({'poses': k, 'psnr': round(s['psnr'], 3), 'ssim': round(s['ssim'], 4), 'lpips': round(s['lpips'], 4),
                 'num_GS': s.get('num_GS'), 'train_min': s.get('train_min')})
print(f"{'poses':>9} | {'PSNR':>7} | {'SSIM':>6} | {'LPIPS':>6} | {'#GS':>9} | min")
for r in rows:
    print(f"{r['poses']:>9} | {r['psnr']:7.2f} | {r['ssim']:6.3f} | {r['lpips']:6.3f} | {r['num_GS'] or 0:9,} | {r['train_min']}")
if 'official_ourimages' in runs:
    gap = runs['ours']['psnr'] - runs['official_ourimages']['psnr']
    print(f"\nPose effect, same images (ours - official): {gap:+.2f} dB  {'✅ poses are equivalent' if abs(gap) <= 0.3 else '⚠️ investigate'}")
    print('Note: "official" is scored against different images (original JPGs), so it is not comparable to the other two.')

# append to the shared metrics.csv (keeps Phase 0's column layout)
CSV = f'{PROJECT}/results/metrics.csv'
header = next(csv.reader(open(CSV))) if os.path.exists(CSV) else \
    ['date', 'run', 'scene', 'res', 'strategy', 'steps', 'num_gaussians', 'psnr', 'ssim', 'lpips', 'gpu', 'mem_gb', 'ply_mb']
new_names = {f'phase1_{SCENE}_1of4_{STEPS//1000}k_{k}poses' for k in runs}
old_rows = list(csv.DictReader(open(CSV))) if os.path.exists(CSV) else []
# remove rows from the double-downscale bug (F1.2) + earlier copies of these runs; relabel Phase 0's real resolution
old_rows = [r for r in old_rows if not r['run'].startswith('phase1_garden_f4_') and r['run'] not in new_names]
for r in old_rows:
    if r['run'] == 'phase0_garden_f8_7k': r['res'] = '~1/64 (bug F1.2)'
with open(CSV, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=header, extrasaction='ignore')
    w.writeheader(); w.writerows(old_rows)
    for k, s in runs.items():
        w.writerow({'date': str(datetime.date.today()), 'run': f'phase1_{SCENE}_1of4_{STEPS//1000}k_{k}poses',
                    'scene': SCENE, 'res': '1/4', 'strategy': 'default', 'steps': STEPS,
                    'num_gaussians': s.get('num_GS'), 'psnr': round(s['psnr'], 3), 'ssim': round(s['ssim'], 4),
                    'lpips': f"{round(s['lpips'], 4)} ({'vgg' if LPIPS else 'alex'})",
                    'gpu': torch.cuda.get_device_name(0), 'mem_gb': s.get('mem', '')})
qa = json.load(open(f'{OUT}/pose_qa.json')); qa['training'] = rows
json.dump(qa, open(f'{OUT}/pose_qa.json', 'w'), indent=1)
print('\nappended to', CSV, '| summary in', f'{OUT}/pose_qa.json')

In [ ]:
# side-by-side eval renders (GT | render) from our-poses run
rd = f"{PROJECT}/results/phase1_{SCENE}_f4_{STEPS//1000}k_ourposes"
for f in sorted(glob.glob(f'{rd}/renders/val_step*_*.png'))[:2]:
    plt.figure(figsize=(16, 5)); plt.imshow(imageio.imread(f)); plt.axis('off'); plt.title(os.path.basename(f)); plt.show()

## ✅ Phase 1 (garden) checklist
- [ ] COLMAP registered > 95 % of images, mean reprojection error < 1 px
- [ ] Rotation error median < 0.5°, center error median < 2 % vs official poses
- [ ] Pose effect (same images, official vs our poses) within ~0.3 dB
- [ ] `pose_comparison.png` + `pose_qa.json` saved on Drive → copy into `docs/` for the README
- [ ] Anything that broke → `docs/failures.md`

**Paste these back into the chat:** the outputs of sections 7, 8 and 11 (plus `timings`).

**Next (Phase 1b):** a *raw video* scene (DL3DV or WayveScenes101): FFmpeg frame extraction → blur filtering → sequential matching → masks for sky and moving objects.